# 05 · KPI Report

**Configuración base:**
- Catálogo: `main`
- Schemas: `loterias_raw`, `loterias_bronze`, `loterias_silver`, `loterias_features`
- Volume crudos: `/Volumes/main/loterias_raw/raw_apuestas/apuestas_partitioned/`

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TABLE_FEATURE = "main.loterias_features.features_apuestas"  # ajusta si usas otro nombre


In [0]:
pdf = spark.table(TABLE_FEATURE).toPandas()

In [0]:
kpis = {
    "total_tx": int(len(pdf)),
    "fraude_rate": round(float(pdf["es_fraude"].mean()), 4),
    "monto_promedio": round(float(pdf["monto"].mean()), 2),
    "monto_mediana": round(float(pdf["monto"].median()), 2),
    "usuarios_unicos": int(pdf["user_id"].nunique()),
    "tx_por_usuario": round(len(pdf) / pdf["user_id"].nunique(), 2),
    "tx_max_usuario": int(pdf["user_id"].value_counts().max()),
}
display(pd.DataFrame([kpis]))

In [0]:
canal_stats = (
    pdf.groupby("canal")
       .agg(tx_total=("tx_id","count"),
            fraude_rate=("es_fraude","mean"),
            monto_promedio=("monto","mean"))
       .reset_index()
)
canal_stats["fraude_rate"] = canal_stats["fraude_rate"].round(4)
canal_stats["monto_promedio"] = canal_stats["monto_promedio"].round(2)
display(canal_stats.sort_values("tx_total", ascending=False))

In [0]:
# ---------- Tasa de fraude por día + anomalías ----------
fraude_dia = (
    pdf.groupby("fecha")["es_fraude"]
       .mean()
       .reset_index()
       .rename(columns={"es_fraude":"fraude_rate"})
       .sort_values("fecha")
)

In [0]:
# rolling para suavizar y z-score para anomalías
win = 7
fraude_dia["roll_mean"] = fraude_dia["fraude_rate"].rolling(win, min_periods=1).mean()
fraude_dia["roll_std"]  = fraude_dia["fraude_rate"].rolling(win, min_periods=2).std()
fraude_dia["z"] = (fraude_dia["fraude_rate"] - fraude_dia["roll_mean"]) / fraude_dia["roll_std"]
fraude_dia["anomaly"] = fraude_dia["z"].abs() > 2.0  # umbral 2σ

display(fraude_dia)

plt.figure(figsize=(10,4))
plt.plot(fraude_dia["fecha"], fraude_dia["fraude_rate"], marker="o")
plt.plot(fraude_dia["fecha"], fraude_dia["roll_mean"])
plt.title("Evolución tasa de fraude (línea: promedio móvil 7d)")
plt.xlabel("fecha"); plt.ylabel("fraude_rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# ---------- Top IPs y Usuarios (con mínimo de transacciones) ----------
MIN_TX = 5
ip_stats = (pdf.groupby("ip")
              .agg(tx=("tx_id","count"), fraude_rate=("es_fraude","mean"))
              .query("tx >= @MIN_TX")
              .sort_values(["fraude_rate","tx"], ascending=[False, False])
              .head(10)
              .reset_index())
ip_stats["fraude_rate"] = ip_stats["fraude_rate"].round(4)
display(ip_stats)

usr_stats = (pdf.groupby("user_id")
               .agg(tx=("tx_id","count"), fraude_rate=("es_fraude","mean"))
               .query("tx >= @MIN_TX")
               .sort_values(["fraude_rate","tx"], ascending=[False, False])
               .head(10)
               .reset_index())
usr_stats["fraude_rate"] = usr_stats["fraude_rate"].round(4)
display(usr_stats)

In [0]:
if "score" in pdf.columns:
    plt.figure(figsize=(6,4))
    plt.hist(pdf["score"], bins=30)
    plt.title("Distribución de score")
    plt.xlabel("score"); plt.ylabel("frecuencia")
    plt.tight_layout()
    plt.show()

    # Lift por deciles (si score existe y hay etiqueta)
    tmp = pdf[["score","es_fraude"]].dropna().copy()
    tmp["decile"] = pd.qcut(tmp["score"], 10, labels=False, duplicates="drop")
    lift = (tmp.groupby("decile")["es_fraude"]
              .mean()
              .sort_index(ascending=False)  # decil 9 = mejores scores
              .reset_index()
              .rename(columns={"es_fraude": "fraude_rate"}))
    display(lift)